# Humedales

> **Contexto de dominio:** [`docs/fuentes/humedales.md`](../../docs/fuentes/humedales.md)  
> **Bloque:** A | **Línea:** `humedales`  
> **Variable principal:** `nivel_agua` (m)  
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/humedales.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "humedales"
VARIABLE = "nivel_agua"
UNIDAD = "m"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `humedales` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "humedales.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/humedales.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/humedales.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "nivel_agua": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: humedales -->

## 1b. Calidad del agua y estado trofico del humedal

El **estado trofico** clasifica el nivel de enriquecimiento por nutrientes. El modelo de **Vollenweider** predice la concentracion de fosforo total en equilibrio:

```
[P] = L / (qs * (1 + tau^0.5))     L = carga de fosforo (mg/m2/ano)
qs = caudal especifico (m/ano)       tau = tiempo de residencia (anos)
```

| Estado trofico | [P] ug/L | Clorofila-a ug/L | OD (min, mg/L) |
|---|---|---|---|
| Oligotrofico | < 10 | < 2 | > 7 |
| Mesotrofico | 10 - 30 | 2 - 8 | 5 - 7 |
| Eutrofico | 30 - 100 | 8 - 25 | 3 - 5 |
| Hipertrofico | > 100 | > 25 | < 3 |

> **Norma:** Res. 196/2006 y Politica Nacional Humedales 2002. OD < 4 mg/L es umbral critico de deterioro (IDEAM).

> **Nota NDWI:** Indice de Agua de Diferencia Normalizada NDWI = (Green - NIR)/(Green + NIR) — valores > 0 indican presencia de agua libre (Sentinel-2).

In [ ]:
# Indicadores fisicoquimicos y estado trofico del humedal (simulados)
np.random.seed(42)
n = len(df)

# Oxigeno Disuelto (OD) mg/L -- desciende con eutrofizacion
od_trend = np.linspace(6.5, 4.2, n)  # deterioro gradual
od_seasonal = 0.8 * np.sin(2*np.pi*np.arange(n)/12)  # mayor OD en sequia
od = np.clip(od_trend + od_seasonal + np.random.normal(0, 0.3, n), 0.5, 12)

# Fosforo total (ug/L) -- modelo Vollenweider simplificado
L_carga = 500       # mg P/m2/ano (carga tipica humedal andino con presion agricola)
qs = 3.0            # m/ano caudal especifico
tau = 0.5           # anos tiempo de residencia
P_eq = L_carga / (qs * (1 + tau**0.5))  # Vollenweider
fosforo = np.clip(
    P_eq + np.linspace(0, 20, n) + np.random.normal(0, 5, n), 5, 200)

# Clorofila-a (ug/L) -- correlacionada con fosforo
clorofila = np.clip(0.18 * fosforo**0.9 + np.random.normal(0, 1.5, n), 0.5, 80)

# NDWI simulado (espejo de agua) -- varia con hidroperiodo
nivel_agua = df['nivel_agua'].values
ndwi = np.clip(-0.1 + 0.4*(nivel_agua - nivel_agua.min())/(nivel_agua.max()-nivel_agua.min())
               + np.random.normal(0, 0.03, n), -0.3, 0.6)

df['od_mgl'] = od.round(2)
df['fosforo_ugl'] = fosforo.round(1)
df['clorofila_ugl'] = clorofila.round(2)
df['ndwi'] = ndwi.round(3)

# Clasificacion estado trofico
def estado_trofico(p):
    if p < 10: return 'Oligotrofico'
    if p < 30: return 'Mesotrofico'
    if p < 100: return 'Eutrofico'
    return 'Hipertrofico'

df['estado_trofico'] = df['fosforo_ugl'].apply(estado_trofico)

print(f'Vollenweider [P]eq = {P_eq:.1f} ug/L ({estado_trofico(P_eq)})')
print(f'OD actual | min={od.min():.2f} max={od.max():.2f} media={od.mean():.2f} mg/L')
print(f'Meses con OD < 4 mg/L (deterioro): {(od < 4).sum()}/{n}')
print(df['estado_trofico'].value_counts())
df[['fecha','nivel_agua','od_mgl','fosforo_ugl','clorofila_ugl','ndwi','estado_trofico']].head()

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `humedales` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_humedales.html",
       title="EDA — Humedales", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "nivel_agua", title="Humedales — nivel_agua (m)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "nivel_agua", period="month")
plt.show()

## 3c. Dashboard calidad del agua — OD, fosforo, NDWI y bioindicadores

Los **macroinvertebrados bentonicos** son bioindicadores de calidad ecologica del agua. El indice **BMWP-Col** (Biological Monitoring Working Party adaptado a Colombia) asigna puntajes a familias de macroinvertebrados segun su sensibilidad:

| BMWP-Col | Calidad | Color |
|---|---|---|
| > 150 | Muy buena | Azul |
| 101 - 150 | Buena | Verde |
| 61 - 100 | Regular | Amarillo |
| 36 - 60 | Mala | Naranja |
| < 35 | Muy mala | Rojo |

> **NDWI** (Sentinel-2): valores > 0 = agua libre; -0.1 a 0 = agua con sedimentos; < -0.1 = vegetacion/suelo seco.

In [ ]:
# Simulacion indice BMWP-Col (macroinvertebrados bentonicos)
np.random.seed(7)
# BMWP-Col desciende al deteriorarse la calidad del agua
bmwp = np.clip(
    np.linspace(95, 45, n) + np.random.normal(0, 8, n), 10, 200)
df['bmwp_col'] = bmwp.round(0).astype(int)

BMWP_COLORS = {
    'Muy buena': '#1a73e8', 'Buena': '#27ae60',
    'Regular': '#f1c40f', 'Mala': '#e67e22', 'Muy mala': '#e74c3c'}

def bmwp_categoria(b):
    if b > 150: return 'Muy buena'
    if b > 100: return 'Buena'
    if b > 60: return 'Regular'
    if b > 35: return 'Mala'
    return 'Muy mala'

df['bmwp_cat'] = df['bmwp_col'].apply(bmwp_categoria)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Panel A: OD en el tiempo
ax = axes[0, 0]
ax.plot(df['fecha'], df['od_mgl'], lw=1.5, color='#2980b9')
ax.axhline(4.0, color='red', ls='--', lw=1.5, label='Umbral critico OD=4 mg/L')
ax.axhline(7.0, color='green', ls='--', lw=1, label='Optimo OD>7 mg/L')
ax.set_title('Oxigeno Disuelto (OD)', fontweight='bold')
ax.set_ylabel('OD (mg/L)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Panel B: Fosforo total + estado trofico
ax = axes[0, 1]
trofico_colors = df['estado_trofico'].map(
    {'Oligotrofico':'#1a73e8','Mesotrofico':'#27ae60',
     'Eutrofico':'#f1c40f','Hipertrofico':'#e74c3c'})
ax.scatter(df['fecha'], df['fosforo_ugl'], c=trofico_colors, s=15, alpha=0.7)
ax.axhline(30, color='orange', ls='--', lw=1.5, label='Limite mesotrofico 30 ug/L')
ax.set_title('Fosforo Total + Estado Trofico (Vollenweider)', fontweight='bold')
ax.set_ylabel('Fosforo (ug/L)'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Panel C: BMWP-Col (macroinvertebrados)
ax = axes[1, 0]
bmwp_c = df['bmwp_cat'].map(BMWP_COLORS)
ax.bar(df['fecha'], df['bmwp_col'], color=bmwp_c, width=20, alpha=0.85)
for thresh, label in [(35,'Muy mala'),(60,'Mala'),(100,'Regular'),(150,'Buena')]:
    ax.axhline(thresh, color='gray', ls='--', lw=0.8)
ax.set_title('BMWP-Col (macroinvertebrados bentonicos)', fontweight='bold')
ax.set_ylabel('Puntaje BMWP-Col'); ax.grid(axis='y', alpha=0.3)

# Panel D: NDWI espejo de agua
ax = axes[1, 1]
ax.plot(df['fecha'], df['ndwi'], lw=1.5, color='#3498db', label='NDWI')
ax.axhline(0.0, color='green', ls='--', lw=1.5, label='Umbral agua libre NDWI=0')
ax.fill_between(df['fecha'], df['ndwi'], 0,
                where=(df['ndwi'] > 0), alpha=0.3, color='#3498db', label='Agua libre')
ax.set_title('NDWI — Espejo de agua (Sentinel-2)', fontweight='bold')
ax.set_ylabel('NDWI'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Dashboard Calidad Ecologica — Humedal (Ley 357/1997 Ramsar)',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

print('Estado trofico actual:', df['estado_trofico'].iloc[-1])
print('BMWP-Col ultimo registro:', df['bmwp_col'].iloc[-1], '(', df['bmwp_cat'].iloc[-1], ')')
print(f'Meses con NDWI>0 (agua libre): {(df["ndwi"]>0).sum()}/{n}')

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `humedales` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para Humedales) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["nivel_agua"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `nivel_agua` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["nivel_agua"], variable="nivel_agua")
if rep.empty:
    print("Sin normas colombianas registradas para 'nivel_agua'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["nivel_agua"], method="linear")
print(f"Faltantes antes: {df["nivel_agua"].isna().sum()} | "
      f"después: {df_clean["nivel_agua"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["nivel_agua"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: humedales -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Hidrología / ecosistemas

- **Hidroperiodo:** ciclo anual de inundación; estacionalidad fuerte → SARIMA o Prophet superan a ML sin lags estacionales.
- **Lag ENSO = 3 meses.**
- **Inventario nacional:** Resolución 157/2004 — ~30,781 humedales registrados en RUNAP/SIAC.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Humedales (`humedales`)
- **Variable analizada:** `nivel_agua` (m)
- **Modelos ejecutados:** SARIMA, PROPHET, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/humedales.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.